In [2]:
!pip install pandas numpy mysql-connector-python streamlit

   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
    --------------------------------------- 0.3/17.7 MB ? eta -:--:--
   ---- ----------------------------------- 1.8/17.7 MB 5.5 MB/s eta 0:00:03
   ------ --------------------------------- 2.9/17.7 MB 5.9 MB/s eta 0:00:03
   --------- ------------------------------ 4.2/17.7 MB 5.7 MB/s eta 0:00:03
   ------------ --------------------------- 5.5/17.7 MB 5.9 MB/s eta 0:00:03
   --------------- ------------------------ 6.8/17.7 MB 6.0 MB/s eta 0:00:02
   ------------------ --------------------- 8.1/17.7 MB 6.0 MB/s eta 0:00:02
   -------------------- ------------------- 9.2/17.7 MB 6.0 MB/s eta 0:00:02
   --------------------- ------------------ 9.4/17.7 MB 5.7 MB/s eta 0:00:02
   --------------------- ------------------ 9.4/17.7 MB 5.7 MB/s eta 0:00:02
   --------------------------- ------------ 12.1/17.7 MB 5.4 MB/s eta 0:00:02
   ----------------

In [3]:
import pandas as pd
import numpy as np
import mysql.connector
from mysql.connector import Error

In [8]:
df = pd.read_csv(r'C:\Users\bhara\OneDrive\Desktop\Guvi\Uber Eats\Uber_Eats_data.csv')
print("Original rows:", df.shape[0])

Original rows: 23193


In [9]:
df.head()

,name,online_order,book_table,rate,votes,phone,location,rest_type,dish_liked,cuisines,approx_cost(for two people),listed_in(type),listed_in(city)
0,Jalsa,Yes,Yes,4.1/5,775,080 42297555\r\n+91 9743772233,Banashankari,Casual Dining,"Pasta, Lunch Buffet, Masala Papad, Paneer Laja...","North Indian, Mughlai, Chinese",800,Buffet,Banashankari
1,Spice Elephant,Yes,No,4.1/5,787,080 41714161,Banashankari,Casual Dining,"Momos, Lunch Buffet, Chocolate Nirvana, Thai G...","Chinese, North Indian, Thai",800,Buffet,Banashankari
2,San Churro Cafe,Yes,No,3.8/5,918,+91 9663487993,Banashankari,"Cafe, Casual Dining","Churros, Cannelloni, Minestrone Soup, Hot Choc...","Cafe, Mexican, Italian",800,Buffet,Banashankari
3,Addhuri Udupi Bhojana,No,No,3.7/5,88,+91 9620009302,Banashankari,Quick Bites,Masala Dosa,"South Indian, North Indian",300,Buffet,Banashankari
4,Grand Village,No,No,3.8/5,166,+91 8026612447\r\n+91 9901210005,Basavanagudi,Casual Dining,"Panipuri, Gol Gappe","North Indian, Rajasthani",600,Buffet,Banashankari


In [10]:
print("Before Cleaning:", df.shape)
df = df.drop_duplicates()
df = df.dropna(subset=['name', 'location', 'rate'])


Before Cleaning: (23193, 13)


In [12]:
#Clean Rating
df['rate'] = df['rate'].astype(str).str.replace('/5','',regex = False).str.strip()
df['rate'] = pd.to_numeric(df['rate'], errors = 'coerce')
df['rate'] = df['rate'].fillna(df['rate'].median())

In [13]:
df['approx_cost(for two people)'] = df['approx_cost(for two people)'].astype(str).str.replace(',', '', regex=False)
df['approx_cost(for two people)'] = pd.to_numeric(df['approx_cost(for two people)'], errors='coerce')
df['approx_cost(for two people)'] = df['approx_cost(for two people)'].fillna(df['approx_cost(for two people)'].median())

In [14]:
# Feature Engineering
df['price_segment'] = pd.cut(df['approx_cost(for two people)'],
                             bins=[0, 400, 800, 1500, float('inf')],
                             labels=['Low', 'Mid', 'Premium', 'Ultra-Premium'])

In [15]:
df['online_order'] = df['online_order'].map({'Yes': 1, 'No': 0, 'Yes ':1}).fillna(0).astype(int)
df['book_table'] = df['book_table'].map({'Yes': 1, 'No': 0, 'Yes ':1}).fillna(0).astype(int)

In [24]:
print("After cleaning:", df.shape)
import os
os.makedirs('data/cleaned', exist_ok=True)
df.to_csv('data/cleaned/restaurants_cleaned.csv',index=False)
df.head()

After cleaning: (23158, 14)


,name,online_order,book_table,rate,votes,phone,location,rest_type,dish_liked,cuisines,approx_cost(for two people),listed_in(type),listed_in(city),price_segment
0,Jalsa,1,1,4.1,775,080 42297555\r\n+91 9743772233,Banashankari,Casual Dining,"Pasta, Lunch Buffet, Masala Papad, Paneer Laja...","North Indian, Mughlai, Chinese",800,Buffet,Banashankari,Mid
1,Spice Elephant,1,0,4.1,787,080 41714161,Banashankari,Casual Dining,"Momos, Lunch Buffet, Chocolate Nirvana, Thai G...","Chinese, North Indian, Thai",800,Buffet,Banashankari,Mid
2,San Churro Cafe,1,0,3.8,918,+91 9663487993,Banashankari,"Cafe, Casual Dining","Churros, Cannelloni, Minestrone Soup, Hot Choc...","Cafe, Mexican, Italian",800,Buffet,Banashankari,Mid
3,Addhuri Udupi Bhojana,0,0,3.7,88,+91 9620009302,Banashankari,Quick Bites,Masala Dosa,"South Indian, North Indian",300,Buffet,Banashankari,Low
4,Grand Village,0,0,3.8,166,+91 8026612447\r\n+91 9901210005,Basavanagudi,Casual Dining,"Panipuri, Gol Gappe","North Indian, Rajasthani",600,Buffet,Banashankari,Mid
